# Idea 5 of the ideas planner: aligned DINO tokens on a deeper shortlist (K = 200 instead of 50)
Fact found first: our DINO layer is ALREADY aligned token-to-token (src/semantic_rerank/dino_rerank.py: the window is cut at the peak, both token grids start at their own corner, score = mean token cosine over valid tokens, layer 8). So the idea reduces to its depth: score 200 coarse peaks instead of 50.
Test (no learning, 6 standard queries per photo): on the photos where the pipeline NEVER puts the right place among its 10 fine-matched candidates in any of the 27 queries (no peak within 4 sigma of the truth), find the true place's rank in the coarse list of 200, and in the DINO-fused order of the first 50 (today) and of all 200. Falsification (preplanned): fewer than a third of the true places reach the fused top 10 -> dead. Control: photos that are retrieved today must not lose the true place from the top 10.
Settings by environment: NUDT_SPLIT, POOL_PHOTOS, SEED_IDS_FILE (csv photo column = never retrieved), CONTROL_N, TAG.


In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)

DATA = os.environ.get("GATE_DATA", "official")
MAP = "new"
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))   # same pool and setup seed as Test_Iteration_3, so NUDT seed 101 / 202 pick the same photos
WIDE_HEADINGS = [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0]
WIDE_SCALES = [0.95, 1.0, 1.05]
K_PER_QUERY, M_PER_QUERY = 50, 10
MATCHER_CONFIDENCE, LOFTR_COARSE_THRESHOLD = 0.22, 0.21
MIN_INLIERS, GAMMA = 8, 1.0
NUDT_SEED = int(os.environ.get("NUDT_SEED", "101"))
N_NUDT = int(os.environ.get("N_NUDT", "300"))
EXCLUDE_SEEDS = [int(s) for s in os.environ.get("EXCLUDE_SEEDS", "").split(",") if s.strip()]
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))
EXCLUDE_N = int(os.environ.get("EXCLUDE_N", "300"))   # size of the excluded seeds' samples (seed 101 picked 300; this reproduces Test_Iteration_3's seed 202 photos exactly)
MATCHER = os.environ.get("MATCHER", "loftr")          # "loftr" (the pipeline) or "xoftr" (MINIMA XoFTR, a swap-in with the same plumbing)
XOFTR_COARSE, XOFTR_FINE = float(os.environ.get("XOFTR_COARSE", "0.3")), float(os.environ.get("XOFTR_FINE", "0.1"))
FRESH_EXCLUDE_POOL = int(os.environ.get("FRESH_EXCLUDE_POOL", "0"))   # >0 (NUDT, with POOL_PHOTOS >= all frames): skip every photo of the usual
# pool of this size (setup seed 7), so the sample is from photos no earlier test has used
EXCLUDE_IDS_FILE = os.environ.get("EXCLUDE_IDS_FILE")   # optional csv with a photo column: never draw these photos (an earlier fresh sample)
PHOTOS_FILE = os.environ.get("PHOTOS_FILE")           # optional csv with a photo column: only these photos
print(MATCHER, DATA, "seed", NUDT_SEED, "n", N_NUDT, "exclude", EXCLUDE_SEEDS, "limit", N_LIMIT)

In [ ]:
import functools, gc, torch
from train.datasets import photo_sets
photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.stage_cache import CachedStagePipeline
from semantic_rerank.dino_rerank import rerank_peaks, MISSING
from coarse_search.heading_queries import query_geometry
from image_processing.map_levels import level_for_height
pipeline = CachedStagePipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=GAMMA, k_peaks=200, m_fine=10, matcher="loftr_minima",
                               min_inliers=MIN_INLIERS, ransac_pixels=None, matcher_confidence=MATCHER_CONFIDENCE, satellite=MAP, map_margin_m=None,
                               loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD, dino_weight=None, headings=WIDE_HEADINGS, scale_factors=WIDE_SCALES, cache_dir=None)
pipeline.setup(POOL_PHOTOS, 7)
test_set = pipeline.test_set
T7 = ROOT / "tests/Test_Iteration_7/results"
TAG = os.environ["TAG"]
never = [int(i) for i in pd.read_csv(os.environ["SEED_IDS_FILE"]).photo]
from train.checks import replay
dump = replay.load_set(T7 / "rich_dump" / TAG)
control = [p for p in dump if p not in never][: int(os.environ.get("CONTROL_N", "10"))]
print(len(never), "never retrieved;", len(control), "controls")


In [ ]:
STANDARD = [(-5.0, 1.0), (0.0, 1.0), (5.0, 1.0), (-5.0, 1.05), (0.0, 1.05), (5.0, 1.05)]
def rank_of_truth(order_peaks, truth, sigma, within=4.0):
    d = np.hypot(order_peaks[:, 1] - truth[0], order_peaks[:, 2] - truth[1]) / sigma
    hit = np.where(d <= within)[0]
    return int(hit[0]) + 1 if len(hit) else 0     # 1-based rank, 0 = not in the list
rows = []
for group, ids in [("never", never), ("control", control)]:
    for pid in ids:
        item = test_set.items[pid]; prior = test_set.priors[pid]; truth = [float(v) for v in test_set.truths[pid]]; sigma = float(test_set.sigmas[pid])
        level = level_for_height(item["rel_height"], pipeline.camera)
        for (h, s) in STANDARD:
            found = pipeline.prepare_query(item, prior, level, query_geometry(h, s))
            short, dense = found["shortlist"], found["dense"]
            coarse_rank = rank_of_truth(short[np.argsort(-short[:, 0], kind="stable")], truth, sigma)
            f50 = rerank_peaks(short[:50], dense[:50], keep=50)
            f200 = rerank_peaks(short, dense, keep=len(short))
            rows.append(dict(group=group, photo=pid, height=item["rel_height"], heading=h, scale=s, coarse_rank=coarse_rank,
                             in_top50=int(0 < coarse_rank <= 50), fused50_rank=rank_of_truth(f50, truth, sigma), fused200_rank=rank_of_truth(f200, truth, sigma)))
        torch.cuda.empty_cache(); gc.collect()
    print(group, "done")
Q = pd.DataFrame(rows); Q.to_csv(T7 / ("idea5_deep_shortlist_%s.csv" % TAG), index=False)
def hit(r, m=10): return (r > 0) & (r <= m)
P = Q.groupby(["group", "photo"]).agg(coarse_any50=("in_top50", "max"),
        fused50_top10=("fused50_rank", lambda r: int(hit(r).any())), fused200_top10=("fused200_rank", lambda r: int(hit(r).any())),
        in_list200=("coarse_rank", lambda r: int((r > 0).any())), best_coarse=("coarse_rank", lambda r: int(r[r > 0].min()) if (r > 0).any() else 0)).reset_index()
P.to_csv(T7 / ("idea5_deep_shortlist_photos_%s.csv" % TAG), index=False)
print(P.groupby("group")[["coarse_any50", "fused50_top10", "fused200_top10", "in_list200"]].agg(["sum", "count"]).to_string())
P
